#### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subseuqent processing. Langchain supports multiple schema types and methods for enforcing structured output

#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x0000023373CCAF90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023373CCBCB0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movie rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x0000023373CCAF90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023373CCBCB0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}

In [4]:
# Output without structure
model.invoke("Provide details about the movie inception")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request:** The user asked for "details about the movie inception". This is a straightforward request for information about the 2010 film "Inception".\n\n2.  **Identify Key Information Needed:** I should provide comprehensive but concise details covering:\n   - Basic info (title, release year, director, genre, runtime)\n   - Plot summary (without major spoilers if possible, or with spoiler warnings)\n   - Main cast\n   - Themes and concepts\n   - Production/Behind the scenes\n   - Reception/Box office/Awards\n   - Legacy/Impact\n   - Notable quotes or scenes (optional but helpful)\n\n3.  **Gather Facts (Internal Knowledge):**\n   - Title: Inception\n   - Release Year: 2010\n   - Director/Writer: Christopher Nolan\n   - Genre: Science fiction, action, heist, thriller\n   - Runtime: ~148 minutes (2h 28m)\n   - Cast: Leonardo DiCaprio (Dom Cobb), Joseph Gordon-Levitt (Arthur), Elliot Page (Ariadne), Tom Hard

In [6]:
model_with_structure.invoke("Provide details about the movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="This year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Provide details about movie Inception"\n   - Key entity: "Inception" (movie)\n   - Required information for the `Movie` function: title, year, director, rating\n\n2.  **Identify Required Parameters for `Movie` function:**\n   - `title`: "Inception"\n   - `year`: 2010\n   - `director`: Christopher Nolan\n   - `rating`: Need to provide a rating out of 10. I know Inception has a very high rating. On IMDb it\'s around 8.8/10. I\'ll use 8.8 as it\'s widely recognized.\n\n3.  **Validate Parameters:**\n   - All required parameters are available or can be reasonably provided based on common knowledge.\n   - `title`: "Inception" (string)\n   - `year`: 2010 (integer)\n   - `director`: "Christopher Nolan" (string)\n   - `rating`: 8.8 (number)\n\n4.  **Construct Function Call:**\n   ```json\n   {\n     "function": "Movie",\n     "parameters": {\n      

#### Nested Structure

In [10]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about movie Tenet")
print(response)

title='Tenet' year=2020 cast=[Actor(name='John David Washington', role='Neil'), Actor(name='Robert Pattinson', role='Ives'), Actor(name='Elizabeth Debicki', role='Kat'), Actor(name='Michael Caine', role='Sir Michael Crosby')] genres=['Action', 'Sci-Fi', 'Thriller'] budget=200.0


#### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [12]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Please provide details about the movie Spider Man: No way home")
response

{'director': 'Jon Watts',
 'rating': 8.2,
 'title': 'Spider-Man: No Way Home',
 'year': 2021}

In [14]:
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about movie Tenet")
response

{'budget': 200000000,
 'cast': [{'name': 'John David Washington', 'role': 'Protagonist'},
  {'name': 'Robert Pattinson', 'role': 'Neil'},
  {'name': 'Elizabeth Debicki', 'role': 'Kat'},
  {'name': 'Michael Caine', 'role': 'Sir Michael'},
  {'name': 'Dimple Kapadia', 'role': 'Priya'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Tenet',
 'year': 2020}

#### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [21]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key),
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the contact info from: John Doe, john@example.com, +919088565659"}]
})

print(result['structured_response'])

name='John Doe' email='john@example.com' phone='+919088565659'


In [22]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str # "The name of the person"
    email: str # "The email address of the person"
    phone: str # "The phone number of the person"

agent = create_agent(
    model=ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key),
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the contact info from: John Doe, john@example.com, +919088565659"}]
})

print(result['structured_response'])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '+919088565659'}


In [23]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str # "The name of the person"
    email: str # "The email address of the person"
    phone: str # "The phone number of the person"

agent = create_agent(
    model=ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key),
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the contact info from: John Doe, john@example.com, +919088565659"}]
})

print(result['structured_response'])

ContactInfo(name='John Doe', email='john@example.com', phone='+919088565659')
